In [5]:

%pip -q install duckdb huggingface_hub

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from IPython.display import display

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN ?
    )
""", [hf_token])

REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("ML-05: DuckDB ready!")


ML-05: DuckDB ready!


In [6]:

# ML-05: Two signal checks on March 2026 data
# These are exploratory associations, not causal evidence.

# Aggregate daily records into page-level observations
pages = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_sum_position) * 1.0 /
            NULLIF(SUM(gsc_impressions), 0) AS avg_position,
        SUM(gsc_clicks) * 1.0 /
            NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM {REL}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 10
""").df()

print("Eligible page-level observations:", len(pages))

# Signal 1: Search volume versus CTR
pages["volume_bucket"] = pd.cut(
    pages["impressions"],
    bins=[9, 50, 200, 1000, np.inf],
    labels=["10-50", "51-200", "201-1000", "1000+"]
)

print("\nSIGNAL 1: Impressions and CTR")
signal1 = pages.groupby(
    "volume_bucket", observed=True
).agg(
    n=("content_hash_id", "size"),
    median_ctr=("ctr", "median"),
    median_position=("avg_position", "median")
).reset_index()

display(signal1)

# Signal 2: Average search position versus CTR
pages["position_bucket"] = pd.cut(
    pages["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "20+"],
    include_lowest=True
)

print("\nSIGNAL 2: Position and CTR")
signal2 = pages.groupby(
    "position_bucket", observed=True
).agg(
    n=("content_hash_id", "size"),
    median_ctr=("ctr", "median"),
    mean_ctr=("ctr", "mean")
).reset_index()

display(signal2)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible page-level observations: 143206

SIGNAL 1: Impressions and CTR


,volume_bucket,n,median_ctr,median_position
0,10-50,27483,0.000000,9.866667
1,51-200,31014,0.000000,13.514306
2,201-1000,39674,0.000000,9.440078
3,1000+,45035,0.001942,6.358646



SIGNAL 2: Position and CTR


,position_bucket,n,median_ctr,mean_ctr
0,1-3,12432,0.001585,0.003955
1,4-10,65446,0.000977,0.003762
2,11-20,26678,0.000000,0.002607
3,20+,38650,0.000000,0.001361


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amarkosovac11/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


### Baseline Signal Audit — March 2026

**Lane:** Refresh / Content Opportunity Scoring

**Unit of analysis:** One anonymized client-content page aggregated across March 2026.

**Eligible pages:** 143,206.

### Signal 1: Search Impressions vs CTR

**Verdict: MIXED**

The median CTR was zero in the first three impression-volume buckets (10-50, 51-200, and 201-1000).

Pages with more than 1,000 impressions had a median CTR of approximately 0.194%.

Higher impression volume may help identify pages with greater exposure, but does not independently establish a content-refresh opportunity.

### Signal 2: Average Position vs CTR

**Verdict: CONFIRMED**

Mean CTR decreased across the position buckets:

- Positions 1-3: 0.396%
- Positions 4-10: 0.376%
- Positions 11-20: 0.261%
- Positions above 20: 0.136%

This supports the observed directional relationship between better search positions and higher CTR.

### Proposed Baseline Rule

Prioritize pages with meaningful search exposure, relatively weak CTR, and average search positions where improvements may be worth investigating.

This is a rule-based review priority score, not a prediction of future ranking improvement.

The proposed action is to review content freshness, relevance, and search snippets before deciding whether to update a page.

**Reason code:** `SEARCH_REVIEW_OPPORTUNITY`

**Action label:** `REVIEW_CONTENT_AND_SNIPPET`


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:

from pathlib import Path

# Keep pages with meaningful search exposure
queue = pages.copy()

# Score is intentionally a simple, transparent baseline.
# Higher impressions = greater exposure.
# Lower CTR at positions 4-20 = potential review opportunity.

queue["baseline_score"] = (
    np.log1p(queue["impressions"])
    * (1 - queue["ctr"].clip(0, 1))
    * queue["avg_position"].between(4, 20).astype(int)
)

queue["reason_code"] = "SEARCH_REVIEW_OPPORTUNITY"
queue["action_label"] = "REVIEW_CONTENT_AND_SNIPPET"

queue = queue.sort_values(
    ["baseline_score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

output_dir = Path("/content/flyrank-ml-internship/work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

queue[[
    "rank", "client_hash_id", "content_hash_id",
    "impressions", "clicks", "avg_position", "ctr",
    "baseline_score", "reason_code", "action_label"
]].to_csv(output_path, index=False)

print("Baseline queue created:", len(queue), "pages")
print("CSV saved to:", output_path)

display(queue[[
    "rank", "content_hash_id", "impressions",
    "avg_position", "ctr", "baseline_score",
    "reason_code", "action_label"
]].head(10))


Baseline queue created: 143206 pages
CSV saved to: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv


,rank,content_hash_id,impressions,avg_position,ctr,baseline_score,reason_code,action_label
0,1,content_e8a52cf3d5988c07,244931.0,15.173490,0.002731,12.374843,SEARCH_REVIEW_OPPORTUNITY,REVIEW_CONTENT_AND_SNIPPET
1,2,content_b99ea6861864dea5,194337.0,4.551516,0.001858,12.154734,SEARCH_REVIEW_OPPORTUNITY,REVIEW_CONTENT_AND_SNIPPET
2,3,content_e7b5dd4dff461ad2,205045.0,4.453174,0.011929,12.085085,SEARCH_REVIEW_OPPORTUNITY,REVIEW_CONTENT_AND_SNIPPET
3,4,content_471d9cabce329a66,164885.0,4.603724,0.002402,11.984158,SEARCH_REVIEW_OPPORTUNITY,REVIEW_CONTENT_AND_SNIPPET
4,5,content_f43118e089ecc69a,139417.0,5.342218,0.001370,11.829004,SEARCH_REVIEW_OPPORTUNITY,REVIEW_CONTENT_AND_SNIPPET
5,6,content_7c6373141eae744a,132593.0,5.948459,0.000626,11.787664,SEARCH_REVIEW_OPPORTUNITY,REVIEW_CONTENT_AND_SNIPPET
6,7,content_66288edeb93b7c4f,137878.0,13.584096,0.005672,11.767012,SEARCH_REVIEW_OPPORTUNITY,REVIEW_CONTENT_AND_SNIPPET
7,8,content_00d4fdf6e48a2d38,126836.0,5.310314,0.003682,11.707393,SEARCH_REVIEW_OPPORTUNITY,REVIEW_CONTENT_AND_SNIPPET
8,9,content_95ff62babbfac9c7,122857.0,4.623098,0.001905,11.696464,SEARCH_REVIEW_OPPORTUNITY,REVIEW_CONTENT_AND_SNIPPET
9,10,content_5e1c049f62e33b11,120175.0,17.773364,0.001398,11.680361,SEARCH_REVIEW_OPPORTUNITY,REVIEW_CONTENT_AND_SNIPPET


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:

# ML-05: Top-10 manual review preparation

top10 = queue.head(10).copy()

def confidence_note(row):
    if row["avg_position"] > 10:
        return "Medium - weaker ranking position"
    return "Medium - ranking is already relatively strong"

def potential_problem(row):
    if row["avg_position"] > 10:
        return (
            "Low CTR could be explained by search intent, "
            "SERP features, or query mix rather than outdated content."
        )
    return (
        "The page may already satisfy user intent. "
        "Low CTR could reflect the search snippet or branded queries."
    )

top10["confidence_note"] = top10.apply(
    confidence_note, axis=1
)

top10["what_could_be_wrong"] = top10.apply(
    potential_problem, axis=1
)

for _, row in top10.iterrows():
    print(f"\nRANK {int(row['rank'])}")
    print(f"Page: {row['content_hash_id']}")
    print(f"Action: {row['action_label']}")
    print(
        f"Reason: {row['impressions']:,.0f} impressions, "
        f"average position {row['avg_position']:.2f}, "
        f"CTR {row['ctr'] * 100:.2f}%."
    )
    print(f"Confidence: {row['confidence_note']}")
    print(f"What could make it wrong: {row['what_could_be_wrong']}")



RANK 1
Page: content_e8a52cf3d5988c07
Action: REVIEW_CONTENT_AND_SNIPPET
Reason: 244,931 impressions, average position 15.17, CTR 0.27%.
Confidence: Medium - weaker ranking position
What could make it wrong: Low CTR could be explained by search intent, SERP features, or query mix rather than outdated content.

RANK 2
Page: content_b99ea6861864dea5
Action: REVIEW_CONTENT_AND_SNIPPET
Reason: 194,337 impressions, average position 4.55, CTR 0.19%.
Confidence: Medium - ranking is already relatively strong
What could make it wrong: The page may already satisfy user intent. Low CTR could reflect the search snippet or branded queries.

RANK 3
Page: content_e7b5dd4dff461ad2
Action: REVIEW_CONTENT_AND_SNIPPET
Reason: 205,045 impressions, average position 4.45, CTR 1.19%.
Confidence: Medium - ranking is already relatively strong
What could make it wrong: The page may already satisfy user intent. Low CTR could reflect the search snippet or branded queries.

RANK 4
Page: content_471d9cabce329a66
A


### Top-10 Review Summary

I reviewed the ten highest-ranked pages generated by the baseline scoring rule.

All ten received the action `REVIEW_CONTENT_AND_SNIPPET` with reason code `SEARCH_REVIEW_OPPORTUNITY`.

**Key observations:**

- All ten pages have substantial search impressions, ranging from 120,175 to 244,931.
- Three pages (ranks 1, 7, and 10) have average search positions worse than 10.
- Seven pages have average positions between approximately 4.45 and 5.95.
- Low CTR does not necessarily indicate outdated content or a need for rewriting.

**Potential weak recommendations:**

Rank 3 has a CTR of approximately 1.19%, which is substantially higher than several other top-ranked pages. Its high priority is driven mainly by impression volume rather than exceptionally weak CTR.

Ranks 2, 4, 5, 6, 8, and 9 already have relatively strong average search positions, so content updates may not be the most appropriate action.

**Conclusion:**

The baseline successfully generates a transparent review queue, but its scoring formula strongly favors high-impression pages.

A human reviewer should inspect search intent, current content quality, and search snippets before recommending changes.

These findings support the need for more careful evaluation and possibly improved scoring in later modeling work.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


### Weak Recommendations and Leakage Check

The baseline identified 143,206 eligible page-level observations and produced a ranked review queue.

**Potential weak recommendations:**

1. A high-impression page may already perform well and may not require a content refresh.
2. Low CTR may reflect query intent, SERP features, or the search snippet rather than outdated content.
3. Pages with average positions near 4-5 may already have relatively strong visibility.
4. A high baseline score does not guarantee that a content update will improve future traffic.
5. This baseline does not directly measure content freshness or quality.

### Leakage Check

The baseline uses March 2026 search-performance metrics aggregated at page level.

It does not use future-period outcomes or the intentionally leaked label feature from ML-04.

The scoring rule does not use FlyRank product flags or future-window labels.

All inputs are assumed to be available after the March reporting window has closed.

### Limitations

This is an exploratory scoring baseline, not a validated predictive ML model.

The scoring formula prioritizes high exposure and lower CTR within a selected position range.

It may systematically favor popular pages over less-visible pages with genuine content problems.

The next stage should compare this transparent baseline against an appropriately validated model.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.